In [79]:
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any, List
import time
from tqdm import tqdm 
import json

In [80]:
import RFA_utils

## with the help of RFA_utils we can use two main function
1. **extract_all_RFA_article_links: Extracts all article links from a given RFA webpage.**
2. scrape_rfa_article: Scrapes an article from the RFA website.


------------
-----------
------------
------------
------------

In [62]:
def loop_article_page(total_page, custom_url, key_code):
    """
    
    """
    return_file = {
        "Data": [],
        "message": "success",
        "response": 200
    }
    All_url_links = {}
    
    try:
        for i in tqdm(range(0, total_page)):
            final_url = custom_url + str(i*15) 
            # found_url_links = RFA_utils.extract_all_RFA_article_links(final_url)
            try:
                found_url_links = RFA_utils.extract_all_RFA_article_links(final_url)
            except Exception as e:
                print(f"Error on page {i}: {e}")
                found_url_links = {"Links": [], "Message": str(e), "Response": 404, "source_url": final_url}
    
            key = key_code + str(i)
            All_url_links[key] = found_url_links
        return_file["Data"] = All_url_links
        return return_file
    
    except Exception as e:
        return_file["Data"] = All_url_links
        return_file["message"] = e
        return_file["response"] = 404
        return return_file

In [13]:
def check_error_in_links(All_url_link, page_code, print_each_error=False):
    """
    
    """

    error_counter = 0
    for page_id in range(1, len(All_url_link)):
        page_key = page_code + str(page_id)
        try:
            All_url_link.get(page_key)
            if  All_url_link.get(page_key)["Response"]!= 200:
                error_counter += 1
                if print_each_error:
                    print(page_key, All_url_link.get(page_key)["message"])
        except Exception as e:
            print(page_key, e)

    print(f"Total error in {page_code}: {error_counter}")

In [14]:
def save_json(path, file_name, data):
    """
    
    """
    with open(path+file_name, "w") as outfile:
        json.dump(data, outfile, indent=4)
        print(f"Successfully saved: {file_name}")

In [15]:
# Saving the final file
path = "./data/"

In [24]:
def compare_with_existing_data(new_data, existing_file_path, tag):
    """
    Compare newly extracted links with existing data to find new articles.
    
    Args:
        new_data (dict): Dictionary containing newly extracted article links
        existing_file_path (str): Path to the existing JSON file
        tag (str): Tag/category of the articles (e.g., "གོང་ས་མཆོག")
        
    Returns:
        dict: Dictionary containing statistics and new article links
    """
    comparison_result = {
        "tag": tag,
        "total_new_links": 0,
        "total_existing_links": 0,
        "new_links": [],
        "message": "Success",
        "response": 200
    }
    
    try:
        # Load existing data
        with open(existing_file_path, 'r', encoding='utf-8') as file:
            existing_data = json.load(file)
        
        # print(existing_data)
        
        # Extract all existing links into a set for faster lookup
        existing_links = set()
        for page_key in existing_data:
            # print(page_key)
            page_links = existing_data[page_key].get("Links", [])
            for link in page_links:
                existing_links.add(link)
        
        comparison_result["total_existing_links"] = len(existing_links)
        
        # Find new links
        new_links = []
        for page_key in new_data.get("Data", {}):
            page_links = new_data["Data"][page_key].get("Links", [])
            for link in page_links:
                if link not in existing_links:
                    new_links.append(link)
        
        comparison_result["total_new_links"] = len(new_links)
        comparison_result["new_links"] = new_links
        
        return comparison_result
    
    except Exception as e:
        comparison_result["message"] = f"Error comparing data: {str(e)}"
        comparison_result["response"] = 500
        return comparison_result

------------
------------
------------



# A. Extracting all Article links from ༸གོང་ས་མཆོག 
- Base url: https://www.rfa.org/tibetan/dalai-lama/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/dalai-lama/story_archive?b_start:int= + str(i)
- Total page:116

In [18]:
total_page = 126 + 1
custom_url= "https://www.rfa.org/tibetan/dalai-lama/story_archive?b_start:int="
article_tag = "གོང་ས་མཆོག"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page གོང་ས་མཆོག 


 99%|█████████▉| 126/127 [11:13<00:05,  5.34s/it]


In [19]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in གོང་ས་མཆོག: 126


In [20]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page གོང་ས་མཆོག : 0


In [ ]:
# all_links

In [26]:
# Path to existing data file
existing_file_path = "./data/RFA_ALL_link_གོང་ས་མཆོག.json"

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

Existing links: 1717
New links found: 140
New articles found:
1. https://www.rfa.org/tibetan/sargyur/hhdl-audience-430-devotees-rfatibetan-03192025061353.html
2. https://www.rfa.org/tibetan/sargyur/dalai-lama-gelong-vow-03182025065159.html
3. https://www.rfa.org/tibetan/sargyur/dalai-lama-audience-robert-thurmon-03172025055501.html
4. https://www.rfa.org/tibetan/sargyur/hhdl-congratulatory-message-cana-pm-mark-carney-03152025051629.html
5. https://www.rfa.org/tibetan/sargyur/hhdl-gelong-ordination-dharamsala-03152025041046.html
6. https://www.rfa.org/tibetan/sargyur/dalai-lama-teaching-jataka-dharamshala-03142025043815.html
7. https://www.rfa.org/tibetan/sargyur/hhdl-chotrul-duechen-2025-03142025015111.html
8. https://www.rfa.org/tibetan/sargyur/dalai-lama-gold-mercury-03132025151440.html
9. https://www.rfa.org/tibetan/sargyur/hhdl-gelong-ordination-2nd-day-51-devotees-03132025041517.html
10. https://www.rfa.org/tibetan/sargyur/hhdl-voice-for-the-voiceles-his-successor-born-free-world-

------------
------------
------------



# B. Extracting all Article links from བོད།
- Base url: https://www.rfa.org/tibetan/tibet/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/tibet/story_archive?b_start:int= + str(i)
- Total page:159

In [27]:
total_page = 179 + 1
custom_url= "https://www.rfa.org/tibetan/tibet/story_archive?b_start:int="
article_tag = "བོད།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)

Page code: Page བོད། 


 99%|█████████▉| 179/180 [16:19<00:05,  5.47s/it]


In [28]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in བོད།: 179


In [29]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page བོད། : 0


In [32]:
# Path to existing data file
file_name = f"./data/RFA_ALL_link_{article_tag}.json"

existing_file_path = file_name

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

Existing links: 2356
New links found: 309
New articles found:
1. https://www.rfa.org/tibetan/sargyur/radio-free-asia-tibet-concern-03192025054313.html
2. https://www.rfa.org/tibetan/sargyur/chinese-government-strict-restrictions-taktsang-lhamo-kirti-ngaba-kirti-monastery-rfatibetan-03192025053757.html
3. https://www.rfa.org/tibetan/tibet/how-tibet-has-remained-under-illegal-occupation-by-the-peoples-republic-of-china-since-1950-at-the-university-of-stirling-in-scotland-03182025092309.html
4. https://www.rfa.org/tibetan/tibet/lobsang-tashi-four-rivers-six-ranges-03172025155035.html
5. https://www.rfa.org/tibetan/sargyur/radio-free-asia-rfa-voa-usagm-executive-order-federal-grants-termination-03152025221057.html
6. https://www.rfa.org/tibetan/indiaandchina/annual-communist-chinas-meetings-03152025131122.html
7. https://www.rfa.org/tibetan/tibet/tibet-lakes-are-surging-03142025144112.html
8. https://www.rfa.org/tibetan/sargyur/china-regulations-model-region-tar-aim-intensify-sinicization-

------------
------------
------------



# C. Extracting all Article links from བཙན་བྱོལ།
- Base url: https://www.rfa.org/tibetan/exile/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/exile/story_archive?b_start:int= + str(i)
- Total page:257

In [38]:
total_page = 295 + 1
custom_url= "https://www.rfa.org/tibetan/exile/story_archive?b_start:int="
article_tag = "བཙན་བྱོལ།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page བཙན་བྱོལ། 


100%|█████████▉| 295/296 [26:04<00:05,  5.30s/it]


In [39]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in བཙན་བྱོལ།: 295


In [40]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page བཙན་བྱོལ། : 0


In [41]:
# Path to existing data file
file_name = f"./data/RFA_ALL_link_{article_tag}.json"

existing_file_path = file_name

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

Existing links: 3842
New links found: 565
New articles found:
1. https://www.rfa.org/tibetan/sargyur/58th-unhrc-session-representative-thinley-choekyi-rfatibetan-tibet-advocacy-03202025062530.html
2. https://www.rfa.org/tibetan/exile/tibet-advocacy-week-in-india-03192025172601.html
3. https://www.rfa.org/tibetan/exile/philadelpia-weekend-tibetan-language-school-03122025161604.html
4. https://www.rfa.org/tibetan/sargyur/rfatibetan-tibetan-parliament-session-9-rfa-voa-03192025065140.html
5. https://www.rfa.org/tibetan/sargyur/rfa-voa-china-us-news-outlets-03172025151636.html
6. https://www.rfa.org/tibetan/sargyur/tibetan-parliament-exile-nine-session-03182025064927.html
7. https://www.rfa.org/tibetan/sargyur/9th-session-of-17th-tibetan-parliament-in-exile-03172025060102.html
8. https://www.rfa.org/tibetan/exile/city-council-and-county-of-santa-barbara-issued-official-proclamations-on-march-10th-and-11th-respectively-unanimously-designating-tibetan-uprising-day-as-tibet-day-03152025092246

------------
------------
------------



# D. Extracting all Article links from འཛམ་གླིང༌།
- Base url: https://www.rfa.org/tibetan/world/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/world/story_archive?b_start:int= + str(i)
- Total page:199

In [42]:
total_page = 233 + 1
custom_url= "https://www.rfa.org/tibetan/world/story_archive?b_start:int="
article_tag = "འཛམ་གླིང༌།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page འཛམ་གླིང༌། 


100%|█████████▉| 233/234 [23:45<00:06,  6.12s/it]


In [43]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in འཛམ་གླིང༌།: 233


In [44]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page འཛམ་གླིང༌། : 0


In [45]:
# Path to existing data file
file_name = f"./data/RFA_ALL_link_{article_tag}.json"

existing_file_path = file_name

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

Existing links: 2954
New links found: 521
New articles found:
1. https://www.rfa.org/tibetan/sargyur/his-holiness-the-dalai-lama-90th-birthday-celebration-cta-rfatibetan-03202025073039.html
2. https://www.rfa.org/tibetan/sargyur/58th-unhrc-session-representative-thinley-choekyi-rfatibetan-tibet-advocacy-03202025062530.html
3. https://www.rfa.org/tibetan/sargyur/rfatibetan-tibetan-parliament-session-9-rfa-voa-03192025065140.html
4. https://www.rfa.org/tibetan/sargyur/rfa-voa-china-us-news-outlets-03172025151636.html
5. https://www.rfa.org/tibetan/sargyur/tibetan-parliament-exile-nine-session-03182025064927.html
6. https://www.rfa.org/tibetan/sargyur/9th-session-of-17th-tibetan-parliament-in-exile-03172025060102.html
7. https://www.rfa.org/tibetan/world/thai-government-handed-over-40-uighurs-to-the-chinese-government-03142025150343.html
8. https://www.rfa.org/tibetan/sargyur/tibet-support-groups-urge-uk-energy-climate-change-minister-concerns-tibet-03142025074858.html
9. https://www.rfa.

------------
------------
------------



# E. Extracting all Article links from རྒྱ་དཀར་ནག
- Base url: https://www.rfa.org/tibetan/indiaandchina/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/indiaandchina/story_archive?b_start:int= + str(i)
- Total page:133

In [47]:
total_page = 144 + 1
custom_url= "https://www.rfa.org/tibetan/indiaandchina/story_archive?b_start:int="
article_tag = "རྒྱ་དཀར་ནག"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page རྒྱ་དཀར་ནག 


 99%|█████████▉| 144/145 [13:57<00:05,  5.82s/it]


In [48]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in རྒྱ་དཀར་ནག: 144


In [49]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page རྒྱ་དཀར་ནག : 0


In [50]:
# Path to existing data file
file_name = f"./data/RFA_ALL_link_{article_tag}.json"

existing_file_path = file_name

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

Existing links: 1968
New links found: 166
New articles found:
1. https://www.rfa.org/tibetan/sargyur/china-military-exercises-near-taiwan-03182025141948.html
2. https://www.rfa.org/tibetan/sargyur/taiwan-based-publisher-li-yanhe-charges-in-china-03182025113019.html
3. https://www.rfa.org/tibetan/sargyur/deepseek-travel-ban-03172025170141.html
4. https://www.rfa.org/tibetan/sargyur/chinese-influencer-must-leave-taiwan-03172025144409.html
5. https://www.rfa.org/tibetan/indiaandchina/a-federal-judge-in-missouri-ruled-that-the-chinese-government-should-be-held-accountable-for-covering-up-the-truth-and-hoarding-protective-supplies-in-the-early-stages-of-the-covid-19-outbreak-03152025093647.html
6. https://www.rfa.org/tibetan/indiaandchina/annual-communist-chinas-meetings-03152025131122.html
7. https://www.rfa.org/tibetan/sargyur/taiwan-says-tougher-stepped-up-china-03132025141432.html
8. https://www.rfa.org/tibetan/sargyur/tibetan-uprising-in-nepal-03112025161924.html
9. https://www.rfa.org

------------ 
------------
------------



# F. Extracting all Article links from སྤྱི་ཚོགས།
- Base url: https://www.rfa.org/tibetan/society/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/society/story_archive?b_start:int= + str(i)
- Total page:241

In [51]:
total_page = 260 + 1
custom_url= "https://www.rfa.org/tibetan/society/story_archive?b_start:int="
article_tag = "སྤྱི་ཚོགས།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page སྤྱི་ཚོགས། 


100%|█████████▉| 260/261 [25:13<00:05,  5.82s/it]


In [52]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in སྤྱི་ཚོགས།: 260


In [53]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page སྤྱི་ཚོགས། : 0


In [54]:
# Path to existing data file
file_name = f"./data/RFA_ALL_link_{article_tag}.json"

existing_file_path = file_name

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

Existing links: 3601
New links found: 285
New articles found:
1. https://www.rfa.org/tibetan/society/tibetan-ability-center-03192025170626.html
2. https://www.rfa.org/tibetan/society/monpa-and-tibetan-new-year-03192025162428.html
3. https://www.rfa.org/tibetan/sargyur/art-institute-chicago-to-return-12th-century-buddha-sculpture-to-nepal-03142025140744.html
4. https://www.rfa.org/tibetan/society/gho-don-will-be-celebrated-in-three-major-locations-in-the-united-states-03132025125006.html
5. https://www.rfa.org/tibetan/society/tibetan-rapper-tenzin-yonten-shares-his-dedication-to-preserving-the-tibetan-language-through-music-03132025105021.html
6. https://www.rfa.org/tibetan/exile/ngawang-thokmey-a-minnesota-based-entrepreneur-involved-in-acting-and-real-estate-industry-03112025092823.html
7. https://www.rfa.org/tibetan/society/middle-way-approach-and-science-03062025142706.html
8. https://www.rfa.org/tibetan/society/dr-dorjee-wangdue-about-the-speciality-of-his-medical-practice-03072025

------------
------------
------------



# G. Extracting all Article links from གསར་འགྱུར།
- Base url: https://www.rfa.org/tibetan/sargyur/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/sargyur/story_archive?b_start:int= + str(i)
- Total page:1941

In [63]:
total_page = 2004 + 1
custom_url= "https://www.rfa.org/tibetan/sargyur/story_archive?b_start:int="
article_tag = "གསར་འགྱུར།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page གསར་འགྱུར། 


 38%|███▊      | 766/2005 [1:26:00<2:58:31,  8.65s/it]

Error on page 765: 'ValueError' object has no attribute 'response'


 38%|███▊      | 767/2005 [1:26:07<2:44:25,  7.97s/it]

Error on page 766: 'ValueError' object has no attribute 'response'


...

Error on page 2002: 'ValueError' object has no attribute 'response'


100%|█████████▉| 2004/2005 [3:43:04<00:07,  7.36s/it]

Error on page 2003: 'ValueError' object has no attribute 'response'


100%|██████████| 2005/2005 [3:43:10<00:00,  6.68s/it]

Error on page 2004: 'ValueError' object has no attribute 'response'


In [64]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in གསར་འགྱུར།: 2005


In [67]:
# check_error_in_links(all_links['Data'], key_code, print_each_error=True)

In [ ]:
# Path to existing data file
file_name = f"./data/RFA_ALL_link_{article_tag}.json"

existing_file_path = file_name

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

### due to error on page no 766 
- we are solved it and updated in the utils and now running from page 766

In [85]:
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any, List
import time

def extract_all_RFA_article_links(url: str) -> Dict[str, Any]:
    """
    Extracts all article links from a given RFA (Radio Free Asia) webpage.

    Args:
    url (str): The URL of the RFA webpage containing article links.

    Returns:
    Dict[str, Any]: A dictionary containing article links and status details.
    """
    headers = {
        "authority": "www.rfa.org",
        "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "accept-encoding": "gzip, deflate, br, zstd",
        "accept-language": "en-US,en;q=0.9,en-IN;q=0.8",
        "cache-control": "max-age=0",
        "cookie": "AMCVS_518ABC7455E462B97F000101%40AdobeOrg=1; s_cc=true; s_sq=%5B%5BB%5D%5D; utag_main=v_id:019169eade56002296e6ea4a443c0507d001b075008f7$_sn:2$_se:6$_ss:0$_st:1724132420582$vapi_domain:rfa.org$ses_id:1724127626756%3Bexp-session$_pn:6%3Bexp-session; AMCV_518ABC7455E462B97F000101%40AdobeOrg=1176715910%7CMCIDTS%7C19955%7CMCMID%7C92058839809215745258174654077801968713%7CMCAID%7CNONE%7CMCOPTOUT-1724137821s%7CNONE%7CvVersion%7C5.4.0",
        "sec-ch-ua": '"Not)A;Brand";v="99", "Microsoft Edge";v="127", "Chromium";v="127"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "document",
        "sec-fetch-mode": "navigate",
        "sec-fetch-site": "cross-site",
        "sec-fetch-user": "?1",
        "upgrade-insecure-requests": "1",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36 Edg/127.0.0.0"
    }
    final_response = {
        "Links": [],
        "Message": "Success",
        "Response": 200,
        "source_url": url
    }
    
    try:
        start_time = time.time()
        response = requests.get(url, headers=headers, timeout=(5, 60-5))
        response.raise_for_status()
        end_time = time.time()
        if end_time - start_time > 50:
            print(f"This URL took more than 50s: {url}")

        soup = BeautifulSoup(response.content, 'html.parser')
        all_articles = soup.find_all("div", class_="teaserimg")
        if not all_articles:
            # page after 766 
            all_articles = soup.find_all("div", class_="sectionteaser archive")
            if not all_articles:
                raise ValueError("Could not find the main article container on the page.")
        
        article_links = []
        for article in all_articles:
            links = article.find("a")
            if links.get("href"):
                article_links.append(links.get("href"))

        final_response["Links"] = article_links
        return final_response
    
    except requests.Timeout:
        final_response["Message"] = "Request timed out"
        final_response["Response"] = 408
        return final_response
    except requests.RequestException as e:
        final_response["Message"] = f"An error occurred while fetching the webpage: {e}"
        final_response["Response"] = getattr(e, 'status_code', 500)
        return final_response
    except ValueError as e:
        final_response["Message"] = f"An error occurred while parsing the webpage: {e}"
        final_response["Response"] = getattr(e, 'status_code', 500)
        return final_response
    except Exception as e:
        final_response["Message"] = f"An unexpected error occurred: {e}"
        final_response["Response"] = 500
        return final_response



In [86]:
def loop_article_page(total_page, custom_url, key_code):
    """
    
    """
    return_file = {
        "Data": [],
        "message": "success",
        "response": 200
    }
    All_url_links = {}
    
    try:
        for i in tqdm(range(766, total_page)):
            final_url = custom_url + str(i*15)
            # found_url_links = RFA_utils.extract_all_RFA_article_links(final_url)
            try:
                found_url_links = extract_all_RFA_article_links(final_url)
            except Exception as e:
                print(f"Error on page {i}: {e}")
                found_url_links = {"Links": [], "Message": str(e), "Response": 404, "source_url": final_url}
    
            key = key_code + str(i)
            All_url_links[key] = found_url_links
        return_file["Data"] = All_url_links
        return return_file
    
    except Exception as e:
        return_file["Data"] = All_url_links
        return_file["message"] = e
        return_file["response"] = 404
        return return_file

In [ ]:
total_page = 2004 + 1
custom_url= "https://www.rfa.org/tibetan/sargyur/story_archive?b_start:int="
article_tag = "གསར་འགྱུར།"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)

Page code: Page གསར་འགྱུར། 


  8%|▊         | 97/1239 [10:09<2:31:06,  7.94s/it]

In [ ]:
test_file = 

In [ ]:
# Path to existing data file
file_name = f"./data/RFA_ALL_link_{article_tag}.json"

existing_file_path = file_name

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_7.2.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_7.2.json", comparison_result)
else:
    print("No new articles found.")